# Celeris Automobili — demand forecasting walkthrough
Reads the outputs of `run_pipeline` (no numbers typed in). Sections: data quality → anomalies →
drivers → backtest → final forecast.

In [ ]:
import json, pandas as pd, numpy as np
pd.set_option('display.width', 200)
q = json.load(open('../reports/data_quality_report.json'))
issues = pd.DataFrame(q['issues'])
print(q['before_after'])
issues.sort_values('rows_affected', ascending=False)[['file','issue_type','classification','rows_affected','action']]

## 2. Anomaly register — errors vs real events

In [ ]:
a = pd.read_csv('../reports/anomaly_register.csv')
a.groupby(['Classification','Anomaly_Type']).size().rename('rows')

## 3. Drivers — which indicator leads which family, at what lag, and is it stable?

In [ ]:
best = pd.read_csv('../data/features/macro_best_lag.csv')
stab = pd.read_csv('../data/features/macro_lag_stability.csv')
display(best.sort_values('Corr', key=abs, ascending=False).head(10))
stab[stab.Indicator=='New_Vehicle_Orders_Index'].pivot(index='Product_Family', columns='Period', values=['Best_Lag','Corr'])

## 4. Rolling-origin backtest (3 origins × 19-month horizon)

In [ ]:
m = json.load(open('../reports/model_metrics.json'))
display(pd.DataFrame(m['overall']).T)
print('P10-P90 coverage', m['interval_P10_P90_coverage'], '| aggregate WAPE', m['aggregate_level_wape'])
pd.DataFrame(m['by_horizon']).T[['wape','wape_snaive','bias']].plot(title='WAPE by horizon', figsize=(9,3));

## 5. Final forecast (Nov 2026 – May 2028)

In [ ]:
fc = pd.read_csv('../data/forecasts/forecast_detail.csv')
sub = pd.read_csv('../submission.csv')
assert len(sub) == 58219 and sub.Units_Sold_Forecast.ge(0).all()
print(m['test_forecast_summary'])
fc.groupby('Date')[['Forecast_P10','Units_Sold_Forecast','Forecast_P90']].sum().plot(figsize=(9,3));

In [ ]:
fc[fc.Adjustment.notna()].groupby(fc.Adjustment.str.split(':').str[0]).size().rename('rows adjusted by rule')